In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# 1. Загрузка данных
train_df = pd.read_csv('../data/train_clean.csv')
test_df = pd.read_csv('../data/test_clean.csv')

print("Структура колонок:", train_df.columns.tolist())
print("Распределение классов в Train:\n", train_df['label'].value_counts())

# 2. Векторизация (N-граммы символов)
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), max_features=10000)
X_train = vectorizer.fit_transform(train_df['query'].astype(str))
y_train = train_df['label']

X_test = vectorizer.transform(test_df['query'].astype(str))
y_test = test_df['label']

# 3. Обучение Baseline-модели
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# 4. Оценка на чистом тесте
y_pred = clf.predict(X_test)
print("--- Результаты на чистом тесте (test_clean) ---")
print(classification_report(y_test, y_pred))

# 5. Оценка на обфусцированном тесте (WAF Evasion Test)
test_obf_df = pd.read_csv('../data/test_obfuscated.csv')
X_test_obf = vectorizer.transform(test_obf_df['query'].astype(str))
y_test_obf = test_obf_df['label']

y_pred_obf = clf.predict(X_test_obf)
print("--- Результаты на обфусцированном тесте (test_obfuscated) ---")
print(classification_report(y_test_obf, y_pred_obf))

Структура колонок: ['query', 'label']
Распределение классов в Train:
 label
0    15622
1     9102
Name: count, dtype: int64
--- Результаты на чистом тесте (test_clean) ---
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      3905
           1       1.00      0.99      0.99      2276

    accuracy                           0.99      6181
   macro avg       1.00      0.99      0.99      6181
weighted avg       0.99      0.99      0.99      6181

--- Результаты на обфусцированном тесте (test_obfuscated) ---
              precision    recall  f1-score   support

           0       0.52      1.00      0.68      3905
           1       1.00      0.68      0.81     11380

    accuracy                           0.76     15285
   macro avg       0.76      0.84      0.75     15285
weighted avg       0.88      0.76      0.78     15285



In [ ]:
# ----------------------------------------------------
# Эксперимент 2: Обучение модели на train_augmented.csv
# ----------------------------------------------------
train_aug_df = pd.read_csv('../data/train_augmented.csv')

print("Распределение классов в Augmented Train:\n", train_aug_df['label'].value_counts())

# 1. Векторизация на обученном аугментированном датасете
vectorizer_aug = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), max_features=10000)
X_train_aug = vectorizer_aug.fit_transform(train_aug_df['query'].astype(str))
y_train_aug = train_aug_df['label']

# 2. Обучение модели
clf_aug = LogisticRegression(max_iter=1000)
clf_aug.fit(X_train_aug, y_train_aug)

# 3. Оценка на чистом тесте
X_test_clean_trans = vectorizer_aug.transform(test_df['query'].astype(str))
y_pred_clean_aug = clf_aug.predict(X_test_clean_trans)

print("\n=== Результаты Augmented Модели на test_clean ===")
print(classification_report(y_test, y_pred_clean_aug))

# 4. Оценка на обфусцированном тесте (WAF Evasion Test)
X_test_obf_trans = vectorizer_aug.transform(test_obf_df['query'].astype(str))
y_pred_obf_aug = clf_aug.predict(X_test_obf_trans)

print("\n=== Результаты Augmented Модели на test_obfuscated ===")
print(classification_report(y_test_obf, y_pred_obf_aug))